**Deficits**

In [4]:
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter
import pandas as pd
import math

# ============================================================
# PATHS
# ============================================================

ROUTE_FILE = Path(r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\sampled\weekday_morning_sampled_merged.rou.xml")
EDGE_FILE  = Path(r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\edgeData_weekday_morning.xml")
TURN_FILE  = Path(r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\turnData_weekday_morning.xml")

OUT_DIR = Path(r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\analysis_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# HELPERS
# ============================================================

def geh(sim, obs):
    if sim + obs == 0:
        return 0.0
    return math.sqrt(2 * (sim - obs) ** 2 / (sim + obs))

def route_weight(elem):
    """
    Try to infer how many vehicles this element represents.
    Works for common SUMO cases:
    - vehicle/personTrip-like single entity -> 1
    - flow with number=...
    - flow with vehsPerHour over finite interval and period
    If unclear, fallback to 1
    """
    tag = elem.tag

    if tag == "vehicle":
        return 1

    if tag == "flow":
        if elem.get("number") is not None:
            return int(float(elem.get("number")))

        begin = float(elem.get("begin", 0))
        end = float(elem.get("end", 0))
        duration = max(0, end - begin)

        if elem.get("period") is not None:
            period = float(elem.get("period"))
            if period > 0:
                return int(round(duration / period))

        if elem.get("vehsPerHour") is not None:
            vph = float(elem.get("vehsPerHour"))
            return int(round(vph * duration / 3600.0))

        if elem.get("probability") is not None:
            # stochastic, exact number unknown
            return 1

    return 1

def parse_route_edges(elem):
    """
    Return list of edges from:
    - <vehicle><route edges="..."/>
    - <flow><route edges="..."/>
    - <route edges="..."/> standalone
    """
    if elem.tag == "route":
        edges = elem.get("edges")
        if edges:
            return edges.split()
        return None

    for child in elem:
        if child.tag == "route":
            edges = child.get("edges")
            if edges:
                return edges.split()

    return None

# ============================================================
# PARSE ROUTES AND COUNT SIMULATED EDGES / TURNS
# ============================================================

sim_edge_counts = Counter()
sim_turn_counts = Counter()   # keys can be (from,to) or (from,via,to) etc.
n_routes = 0
n_vehicles_equivalent = 0

route_root = ET.parse(ROUTE_FILE).getroot()

for elem in route_root:
    if elem.tag not in {"vehicle", "flow", "route"}:
        continue

    edges = parse_route_edges(elem)
    if not edges or len(edges) == 0:
        continue

    w = route_weight(elem)
    n_routes += 1
    n_vehicles_equivalent += w

    # edge counts
    for e in edges:
        sim_edge_counts[e] += w

    # turn / edgeRelation counts
    # count consecutive edge pairs
    for i in range(len(edges) - 1):
        sim_turn_counts[(edges[i], edges[i + 1])] += w

    # also count longer relations for safety:
    # (from, via, to) represented as tuple of all involved edges
    # e.g. if relation spans 3 edges
    for L in range(3, min(6, len(edges) + 1)):  # up to 5-edge relations
        for i in range(len(edges) - L + 1):
            tup = tuple(edges[i:i + L])
            sim_turn_counts[tup] += w

print(f"Parsed route elements: {n_routes}")
print(f"Vehicle-equivalent count represented: {n_vehicles_equivalent}")

# ============================================================
# PARSE MEASURED EDGE COUNTS
# ============================================================

edge_rows = []
edge_root = ET.parse(EDGE_FILE).getroot()

for interval in edge_root.findall(".//interval"):
    begin = float(interval.get("begin", 0))
    end = float(interval.get("end", 0))

    for edge in interval.findall("edge"):
        edge_id = edge.get("id")
        if edge_id is None:
            continue

        measured = edge.get("entered")
        if measured is None:
            measured = edge.get("count")
        if measured is None:
            measured = edge.get("value")
        if measured is None:
            continue

        measured = float(measured)
        simulated = float(sim_edge_counts.get(edge_id, 0))
        deficit = measured - simulated

        edge_rows.append({
            "type": "edge",
            "begin": begin,
            "end": end,
            "location": edge_id,
            "from_edge": edge_id,
            "to_edge": None,
            "relation_tuple": (edge_id,),
            "measured_count": measured,
            "simulated_count": simulated,
            "deficit": deficit,
            "underflow": deficit > 0,
            "GEH": geh(simulated, measured),
        })

df_edges = pd.DataFrame(edge_rows)

# ============================================================
# PARSE MEASURED TURN COUNTS / EDGE RELATIONS
# ============================================================

turn_rows = []
turn_root = ET.parse(TURN_FILE).getroot()

for interval in turn_root.findall(".//interval"):
    begin = float(interval.get("begin", 0))
    end = float(interval.get("end", 0))

    for rel in interval.findall("edgeRelation"):
        from_edge = rel.get("from")
        to_edge = rel.get("to")
        via = rel.get("via")

        if from_edge is None or to_edge is None:
            continue

        measured = rel.get("count")
        if measured is None:
            measured = rel.get("entered")
        if measured is None:
            measured = rel.get("value")
        if measured is None:
            continue

        measured = float(measured)

        if via:
            # support multi-edge relation like from="A" via="B C" to="D"
            via_edges = via.split()
            relation_tuple = tuple([from_edge] + via_edges + [to_edge])
        else:
            relation_tuple = (from_edge, to_edge)

        simulated = float(sim_turn_counts.get(relation_tuple, 0))
        deficit = measured - simulated

        turn_rows.append({
            "type": "turn",
            "begin": begin,
            "end": end,
            "location": " -> ".join(relation_tuple),
            "from_edge": from_edge,
            "to_edge": to_edge,
            "relation_tuple": relation_tuple,
            "measured_count": measured,
            "simulated_count": simulated,
            "deficit": deficit,
            "underflow": deficit > 0,
            "GEH": geh(simulated, measured),
        })

df_turns = pd.DataFrame(turn_rows)

# ============================================================
# COMBINE AND EXPORT
# ============================================================

df_all = pd.concat([df_edges, df_turns], ignore_index=True)

# only underflows
df_underflow = df_all[df_all["underflow"]].copy()
df_underflow = df_underflow.sort_values(
    by=["deficit", "GEH", "measured_count"],
    ascending=[False, False, False]
)

# summary
print("\n==================== SUMMARY ====================")
print(f"Total measured locations: {len(df_all)}")
print(f"Underflow locations: {len(df_underflow)}")

if len(df_underflow) > 0:
    print("\nTop underflows:")
    print(df_underflow[[
        "type", "location", "measured_count", "simulated_count", "deficit", "GEH"
    ]].head(20).to_string(index=False))
else:
    print("No underflows found.")

# save files
all_csv = OUT_DIR / "routesampler_all_locations_comparison.csv"
underflow_csv = OUT_DIR / "routesampler_underflow_locations.csv"

df_all.to_csv(all_csv, index=False, encoding="utf-8-sig")
df_underflow.to_csv(underflow_csv, index=False, encoding="utf-8-sig")

print(f"\nSaved all comparison file to:\n{all_csv}")
print(f"Saved underflow file to:\n{underflow_csv}")

Parsed route elements: 3057
Vehicle-equivalent count represented: 3057

==================== SUMMARY ====================
Total measured locations: 50
Underflow locations: 5

Top underflows:
type     location  measured_count  simulated_count  deficit      GEH
edge   -237921040          1098.0            872.0    226.0 7.200959
edge  -14811383#2           177.0             96.0     81.0 6.932960
edge  -13831317#2           179.0            102.0     77.0 6.496098
edge   26202222#0           260.0            223.0     37.0 2.380911
edge -202157700#5           463.0            442.0     21.0 0.987211

Saved all comparison file to:
C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\analysis_outputs\routesampler_all_locations_comparison.csv
Saved underflow file to:
C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\analysis_outputs\routesampler_underflow_locations.csv
